# Extract Participating Entities from GSP Documents

This notebook uses Hugging Face question-answering models to identify entities (organizations, agencies, consultants, etc.) that participated in developing GSP planning documents.

In [26]:
# Install required packages if needed
# !pip install transformers torch pandas numpy rpy2

In [27]:
import pandas as pd
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
import torch
from pathlib import Path
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Try to import rpy2, but provide alternative if it fails
try:
    import os
    # Set R_HOME if R is installed in a non-standard location
    # Uncomment and adjust the path below if needed:
    # os.environ['R_HOME'] = '/usr/local/lib/R'  # or wherever R is installed
    
    import rpy2.robjects as robjects
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.conversion import localconverter
    RPY2_AVAILABLE = True
    print("rpy2 imported successfully")
except ImportError as e:
    print(f"Warning: Could not import rpy2: {e}")
    print("Will need to use alternative method to load RDS file")
    RPY2_AVAILABLE = False

rpy2 imported successfully


## Configuration

In [29]:
# Model configuration
MODEL_NAME = "deepset/roberta-base-squad2"  # Good for question answering
# Alternative models to try:
# MODEL_NAME = "distilbert-base-cased-distilled-squad"
# MODEL_NAME = "bert-large-uncased-whole-word-masking-finetuned-squad"

# Questions to extract participating entities
ENTITY_QUESTIONS = [
    "Who prepared this document?",
    "Who prepared this groundwater sustainability plan?",
    "Who is the preparer of this plan?",
    "Which organizations participated in developing this plan?",
    "Who are the authors of this document?",
    "Which consultants were involved in creating this plan?",
    "What agencies contributed to this document?",
    "Who facilitated the planning process?",
    "Which engineering firms worked on this plan?",
    "Who provided technical support for this document?",
    "Which company prepared this groundwater sustainability plan?",
    "What firm developed this plan?"
]

# Processing parameters
CONFIDENCE_THRESHOLD = 0.1  # Minimum confidence score to keep an answer
MAX_CONTEXT_LENGTH = 2000  # Characters to process at once
OUTPUT_DIR = Path("../../data_products/entity_extraction/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load the QA Model

In [31]:
# Check if CUDA is available
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'CUDA' if device == 0 else 'CPU'}")

# Suppress widget warnings
import os
os.environ['JUPYTER_WIDGETS_ECHO'] = '0'

# Initialize the QA pipeline
print(f"Loading model: {MODEL_NAME}")
print("This may take a moment on first run as the model downloads...")

try:
    qa_pipeline = pipeline(
        "question-answering",
        model=MODEL_NAME,
        device=device
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Trying with CPU...")
    qa_pipeline = pipeline(
        "question-answering",
        model=MODEL_NAME,
        device=-1  # Force CPU
    )
    print("Model loaded successfully on CPU!")

Using device: CPU
Loading model: deepset/roberta-base-squad2
This may take a moment on first run as the model downloads...


Device set to use cpu


Model loaded successfully!


## Load and Process Text Files

In [ ]:
pwd

In [34]:
# Load raw text files directly
import os
from pathlib import Path

# Define paths
TEXT_FILES_DIR = Path("../../../data/Multipurpose_Files/portal_files/")
PAGE_DELIMITER = "<<PAGE_BREAK>>"

# Function to read and parse text files
def read_portal_file(filepath):
    """Read a portal text file and split into pages."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        # Split by page delimiter
        pages = content.split(PAGE_DELIMITER)
        
        # Extract GSP ID from filename (assuming format like "0001_document.txt")
        filename = os.path.basename(filepath)
        gsp_id = filename.split('_')[0] if '_' in filename else filename.split('.')[0]
        
        return {
            'gsp_id': gsp_id,
            'filename': filename,
            'filepath': str(filepath),
            'pages': pages,
            'full_text': content,
            'page_count': len(pages)
        }
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None

# Find all text files
text_files = list(TEXT_FILES_DIR.glob("*.txt"))
print(f"Found {len(text_files)} text files")

# Read all files
documents = []
for filepath in tqdm(text_files, desc="Reading text files"):
    doc = read_portal_file(filepath)
    if doc:
        documents.append(doc)

print(f"\\nSuccessfully loaded {len(documents)} documents")
print(f"Total pages: {sum(doc['page_count'] for doc in documents)}")

# Show sample
if documents:
    sample = documents[0]
    print(f"\\nSample document:")
    print(f"  GSP ID: {sample['gsp_id']}")
    print(f"  Filename: {sample['filename']}")
    print(f"  Pages: {sample['page_count']}")
    print(f"  First page preview: {sample['pages'][0][:200]}...")

Found 158 text files


Reading text files: 100%|█████████████████████| 158/158 [00:01<00:00, 86.42it/s]

\nSuccessfully loaded 158 documents
Total pages: 218156
\nSample document:
  GSP ID: v1
  Filename: v1_gsp_num_id_0007.txt
  Pages: 731
  First page preview: Groundwater Sustainability Plan


             Final
         January 2020




              001
...


def extract_entities_from_text(text, questions, qa_pipeline, confidence_threshold=CONFIDENCE_THRESHOLD, top_k=5):
    """
    Extract entities from text using question answering.
    
    Args:
        text: Context text to search
        questions: List of questions to ask
        qa_pipeline: Hugging Face QA pipeline
        confidence_threshold: Minimum confidence score
        top_k: Maximum number of answers per question
    """
    if not text or len(text.strip()) == 0:
        return []
    
    # Truncate text if too long
    if len(text) > MAX_CONTEXT_LENGTH:
        text = text[:MAX_CONTEXT_LENGTH]
    
    extracted_entities = []
    
    for question in questions:
        try:
            # Get multiple answers from the model
            results = qa_pipeline(
                question=question,
                context=text,
                max_answer_len=100,
                handle_impossible_answer=True,
                top_k=top_k  # Request multiple answers
            )
            
            # Handle both single result and list of results
            if not isinstance(results, list):
                results = [results]
            
            # Add all answers that meet confidence threshold
            for result in results:
                if result['score'] > confidence_threshold:
                    # Check if this answer is substantially different from existing ones
                    answer_text = result['answer'].strip()
                    
                    # Skip if this exact answer already exists for this question
                    existing_answers = [e['answer'] for e in extracted_entities if e['question'] == question]
                    if answer_text not in existing_answers:
                        extracted_entities.append({
                            'question': question,
                            'answer': answer_text,
                            'confidence': result['score'],
                            'start': result['start'],
                            'end': result['end']
                        })
                
        except Exception as e:
            print(f"Error processing question '{question}': {str(e)}")
            continue
    
    return extracted_entities


def extract_entities_sliding_window(text, questions, qa_pipeline, confidence_threshold=CONFIDENCE_THRESHOLD, 
                                   window_size=1000, stride=500):
    """
    Extract entities using sliding window approach to find multiple mentions.
    
    Args:
        text: Full text to search
        questions: List of questions
        qa_pipeline: QA pipeline
        confidence_threshold: Minimum confidence
        window_size: Size of text window
        stride: How much to slide the window
    """
    if not text or len(text.strip()) == 0:
        return []
    
    all_entities = []
    text_length = len(text)
    
    # Slide through the text
    for start in range(0, min(text_length, MAX_CONTEXT_LENGTH * 3), stride):
        end = min(start + window_size, text_length)
        window_text = text[start:end]
        
        # Extract entities from this window
        entities = extract_entities_from_text(
            window_text, questions, qa_pipeline, 
            confidence_threshold, top_k=3
        )
        
        # Adjust positions to account for window offset
        for entity in entities:
            entity['start'] += start
            entity['end'] += start
            entity['window_start'] = start
            entity['window_end'] = end
        
        all_entities.extend(entities)
    
    # Deduplicate based on answer text and merge confidence scores
    unique_entities = {}
    for entity in all_entities:
        key = (entity['question'], entity['answer'].lower().strip())
        if key in unique_entities:
            # Keep the one with higher confidence
            if entity['confidence'] > unique_entities[key]['confidence']:
                unique_entities[key] = entity
        else:
            unique_entities[key] = entity
    
    return list(unique_entities.values())


def process_gsp_document(doc, focus_on_beginning=True, max_pages=10, use_sliding_window=False):
    """
    Process a single GSP document.
    
    Args:
        doc: Document dictionary with 'gsp_id', 'pages', etc.
        focus_on_beginning: If True, focus on first pages where authorship is typically found
        max_pages: Maximum number of pages to process
        use_sliding_window: If True, use sliding window approach for better coverage
    """
    results = {'gsp_id': doc['gsp_id'], 'filename': doc['filename'], 'entities': []}
    
    # Determine which pages to process
    if focus_on_beginning:
        # Focus on beginning pages where authorship info is typically found
        pages_to_process = doc['pages'][:max_pages]
        combined_text = ' '.join(pages_to_process)
    else:
        # Use full document
        combined_text = doc['full_text']
    
    # Extract entities
    if use_sliding_window:
        entities = extract_entities_sliding_window(combined_text, ENTITY_QUESTIONS, qa_pipeline)
    else:
        # Truncate if too long
        if len(combined_text) > MAX_CONTEXT_LENGTH * 2:
            combined_text = combined_text[:MAX_CONTEXT_LENGTH * 2]
        entities = extract_entities_from_text(combined_text, ENTITY_QUESTIONS, qa_pipeline, top_k=5)
    
    results['entities'] = entities
    
    return results

In [36]:
def extract_entities_from_text(text, questions, qa_pipeline, confidence_threshold=CONFIDENCE_THRESHOLD):
    """
    Extract entities from text using question answering.
    """
    if not text or len(text.strip()) == 0:
        return []
    
    # Truncate text if too long
    if len(text) > MAX_CONTEXT_LENGTH:
        text = text[:MAX_CONTEXT_LENGTH]
    
    extracted_entities = []
    
    for question in questions:
        try:
            # Get answer from the model
            result = qa_pipeline(
                question=question,
                context=text,
                max_answer_len=100,
                handle_impossible_answer=True
            )
            
            # Check if answer meets confidence threshold
            if result['score'] > confidence_threshold:
                extracted_entities.append({
                    'question': question,
                    'answer': result['answer'],
                    'confidence': result['score'],
                    'start': result['start'],
                    'end': result['end']
                })
                
        except Exception as e:
            print(f"Error processing question '{question}': {str(e)}")
            continue
    
    return extracted_entities


def process_gsp_document(doc, focus_on_beginning=True, max_pages=10):
    """
    Process a single GSP document.
    
    Args:
        doc: Document dictionary with 'gsp_id', 'pages', etc.
        focus_on_beginning: If True, focus on first pages where authorship is typically found
        max_pages: Maximum number of pages to process
    """
    results = {'gsp_id': doc['gsp_id'], 'filename': doc['filename'], 'entities': []}
    
    # Determine which pages to process
    if focus_on_beginning:
        # Focus on beginning pages where authorship info is typically found
        pages_to_process = doc['pages'][:max_pages]
        combined_text = ' '.join(pages_to_process)
    else:
        # Use full document
        combined_text = doc['full_text']
    
    # Truncate if still too long
    if len(combined_text) > MAX_CONTEXT_LENGTH * 2:
        combined_text = combined_text[:MAX_CONTEXT_LENGTH * 2]
    
    # Extract entities
    entities = extract_entities_from_text(combined_text, ENTITY_QUESTIONS, qa_pipeline)
    results['entities'] = entities
    
    return results

# Process documents
print(f"Processing {len(documents)} GSP documents...")

# Process each GSP document
all_results = []

# Process a subset first for testing
test_subset = documents[:5]  # Process first 5 documents

# Choose approach: 
# Option 1: Standard approach with top_k (faster, good for most cases)
print("Using standard approach with multiple answers per question...")
for doc in tqdm(test_subset, desc="Processing GSPs"):
    result = process_gsp_document(doc, focus_on_beginning=True, max_pages=10, use_sliding_window=False)
    all_results.append(result)

# Option 2: Sliding window approach (slower, but better coverage)
# print("Using sliding window approach for comprehensive extraction...")
# for doc in tqdm(test_subset, desc="Processing GSPs"):
#     result = process_gsp_document(doc, focus_on_beginning=True, max_pages=10, use_sliding_window=True)
#     all_results.append(result)

print(f"\\nProcessed {len(all_results)} documents")

# Show sample results
for i, result in enumerate(all_results[:3]):
    print(f"\\nDocument {i+1}: {result['filename']}")
    print(f"  Found {len(result['entities'])} entity mentions")
    
    # Group by question to show multiple answers
    if result['entities']:
        from collections import defaultdict
        by_question = defaultdict(list)
        for entity in result['entities']:
            by_question[entity['question']].append(entity)
        
        # Show first 2 questions with their answers
        for j, (question, answers) in enumerate(list(by_question.items())[:2]):
            print(f"\\n  Q: {question}")
            for answer in answers[:3]:  # Show up to 3 answers per question
                print(f"    A: {answer['answer']} (confidence: {answer['confidence']:.3f})")

In [ ]:
# Process documents with combined approach
print(f"Processing {len(documents)} GSP documents with combined QA + pattern extraction...")

# Process each GSP document
combined_results = []

# Process a subset first for testing
test_subset = documents[:5]  # Process first 5 documents

for doc in tqdm(test_subset, desc="Processing GSPs"):
    result = {'gsp_id': doc['gsp_id'], 'filename': doc['filename'], 'entities': []}
    
    # Get text from first 10 pages
    pages_to_process = doc['pages'][:10]
    combined_text = ' '.join(pages_to_process)
    
    # Use combined extraction
    entities = extract_entities_combined(combined_text[:5000], qa_pipeline, ENTITY_QUESTIONS)
    result['entities'] = entities
    combined_results.append(result)

print(f"\\nProcessed {len(combined_results)} documents")

# Show detailed results
for i, result in enumerate(combined_results[:3]):
    print(f"\\nDocument {i+1}: {result['filename']}")
    print(f"  Found {len(result['entities'])} unique entities")
    
    # Group by method
    qa_entities = [e for e in result['entities'] if e['method'] in ['qa', 'both']]
    pattern_entities = [e for e in result['entities'] if e['method'] in ['pattern', 'both']]
    
    print(f"  - QA-based: {len(qa_entities)} entities")
    print(f"  - Pattern-based: {len(pattern_entities)} entities")
    
    # Show all entities sorted by confidence
    sorted_entities = sorted(result['entities'], key=lambda x: x['confidence'], reverse=True)
    print("\\n  All entities found:")
    for entity in sorted_entities[:15]:  # Show up to 15
        method_tag = f"[{entity['method']}]"
        print(f"    {method_tag:10} {entity['entity'][:60]} (conf: {entity['confidence']:.3f})")

In [45]:
# Process documents
print(f"Processing {len(documents)} GSP documents...")

# Process each GSP document
all_results = []

# Process a subset first for testing
test_subset = documents[:5]  # Process first 5 documents

for doc in tqdm(test_subset, desc="Processing GSPs"):
    # Focus on first 10 pages where authorship info is typically found
    result = process_gsp_document(doc, focus_on_beginning=True, max_pages=10)
    all_results.append(result)

print(f"\\nProcessed {len(all_results)} documents")

# Show sample results
for i, result in enumerate(all_results[:3]):
    print(f"\\nDocument {i+1}: {result['filename']}")
    print(f"  Found {len(result['entities'])} entities")
    if result['entities']:
        for entity in result['entities'][:2]:  # Show first 2 entities
            print(f"    Q: {entity['question']}")
            print(f"    A: {entity['answer']} (confidence: {entity['confidence']:.3f})")

Processing 158 GSP documents...


Processing GSPs: 100%|████████████████████████████| 5/5 [00:15<00:00,  3.08s/it]

\nProcessed 5 documents
\nDocument 1: v1_gsp_num_id_0007.txt
  Found 12 entities
    Q: Who prepared this document?
    A: Provost & Pritchard
Consulting Group (confidence: 0.387)
    Q: Who prepared this groundwater sustainability plan?
    A: Provost & Pritchard
Consulting Group (confidence: 0.396)
\nDocument 2: v1_gsp_num_id_0013.txt
  Found 12 entities
    Q: Who prepared this document?
    A:  (confidence: 0.342)
    Q: Who prepared this groundwater sustainability plan?
    A:  (confidence: 0.210)
\nDocument 3: v1_gsp_num_id_0012.txt
  Found 11 entities
    Q: Who prepared this document?
    A: Davids Engineering (confidence: 0.256)
    Q: Who is the preparer of this plan?
    A: Sedi (confidence: 0.342)


In [47]:
result['entities']

[{'question': 'Who prepared this document?',
  'answer': 'Davids Engineering',
  'confidence': 0.2559719681739807,
  'start': 346,
  'end': 364},
 {'question': 'Who is the preparer of this plan?',
  'answer': 'Sedi',
  'confidence': 0.3421953022480011,
  'start': 1239,
  'end': 1243},
 {'question': 'Which organizations participated in developing this plan?',
  'answer': '',
  'confidence': 0.27950868010520935,
  'start': 0,
  'end': 0},
 {'question': 'Who are the authors of this document?',
  'answer': '',
  'confidence': 0.33930379152297974,
  'start': 0,
  'end': 0},
 {'question': 'Which consultants were involved in creating this plan?',
  'answer': '',
  'confidence': 0.35710254311561584,
  'start': 0,
  'end': 0},
 {'question': 'What agencies contributed to this document?',
  'answer': '',
  'confidence': 0.28160494565963745,
  'start': 0,
  'end': 0},
 {'question': 'Who facilitated the planning process?',
  'answer': '',
  'confidence': 0.5031889081001282,
  'start': 0,
  'end': 0

## Consolidate and Clean Results

In [ ]:
# Convert results to DataFrame
entity_records = []

for result in all_results:
    gsp_id = result['gsp_id']
    for entity in result['entities']:
        entity_records.append({
            'gsp_id': gsp_id,
            'question': entity['question'],
            'entity': entity['answer'],
            'confidence': entity['confidence']
        })

entities_df = pd.DataFrame(entity_records)

if len(entities_df) > 0:
    # Clean and standardize entity names
    entities_df['entity_clean'] = entities_df['entity'].str.strip().str.title()
    
    # Remove very short entities (likely errors)
    entities_df = entities_df[entities_df['entity_clean'].str.len() > 2]
    
    # Group by GSP and entity to consolidate duplicates
    consolidated = entities_df.groupby(['gsp_id', 'entity_clean']).agg({
        'confidence': 'max',
        'question': lambda x: '; '.join(x.unique())
    }).reset_index()
    
    consolidated = consolidated.rename(columns={'entity_clean': 'entity'})
    consolidated = consolidated.sort_values(['gsp_id', 'confidence'], ascending=[True, False])
    
    print(f"\nExtracted {len(consolidated)} unique entities from {consolidated['gsp_id'].nunique()} documents")
    print("\nSample results:")
    print(consolidated.head(10))
else:
    print("No entities extracted")
    consolidated = pd.DataFrame()

## Visualize Results

In [ ]:
if len(consolidated) > 0:
    # Summary statistics
    print("\nExtraction Summary:")
    print(f"Total unique entities: {consolidated['entity'].nunique()}")
    print(f"Average entities per GSP: {consolidated.groupby('gsp_id').size().mean():.1f}")
    print(f"Average confidence score: {consolidated['confidence'].mean():.3f}")
    
    # Most common entities
    print("\nMost frequently mentioned entities:")
    entity_counts = consolidated['entity'].value_counts().head(10)
    for entity, count in entity_counts.items():
        print(f"  {entity}: {count} GSPs")

## Process All Documents (Full Run)

In [ ]:
# Uncomment to process all documents
# all_results_full = []
# 
# for gsp_id in tqdm(gsp_ids, desc="Processing all GSPs"):
#     gsp_docs = documents[documents['gsp_id'] == gsp_id]
#     result = process_gsp_document(gsp_docs, gsp_id, focus_sections=['admin'])
#     all_results_full.append(result)
# 
# # Save raw results
# with open(OUTPUT_DIR / 'raw_entity_extraction.pkl', 'wb') as f:
#     pickle.dump(all_results_full, f)

## Save Results

In [ ]:
# Enhanced pattern-based extraction for lists of organizations
import re

def extract_entities_enhanced(text):
    """
    Extract entities using multiple strategies, especially for lists.
    """
    if not text:
        return []
    
    entities = []
    
    # Strategy 1: Look for sections with headers like "Prepared by:", "Authors:", etc.
    prep_patterns = [
        # Headers followed by content
        r"(?:prepared\s+by|authors?|preparers?|developed\s+by|written\s+by|submitted\s+by)[:\s]+([^\n]+(?:\n(?!\n)[^\n]+)*)",
        # List patterns after headers
        r"(?:participating\s+(?:organizations?|agencies)|contributors?|team\s+members?)[:\s]+([^\n]+(?:\n(?!\n)[^\n]+)*)",
        # Consultant/firm patterns
        r"(?:consultant|consulting\s+firm|engineering\s+firm)[:\s]+([^\n]+)",
    ]
    
    for pattern in prep_patterns:
        matches = re.finditer(pattern, text, re.IGNORECASE | re.MULTILINE)
        for match in matches:
            content = match.group(1).strip()
            # Check if this is a list (has newlines or bullet points)
            if '\n' in content or '•' in content or '■' in content or '-' in content.strip()[0:2]:
                # Extract individual items from the list
                items = extract_list_items(content)
                entities.extend(items)
            else:
                entities.append(content)
    
    # Strategy 2: Look for organization name patterns
    org_patterns = [
        # Common organization suffixes
        r"([A-Z][A-Za-z\s&,]+(?:Inc\.|LLC|Corp\.|Corporation|Associates|Consulting|Engineers|Engineering|Agency|Department|District|Authority|Services|Group|Company|Consultants|Partnership|LLP))",
        # Government agencies
        r"([A-Z][A-Za-z\s]+(?:Department|Agency|District|Authority|Board|Commission|Office)\s+of\s+[A-Za-z\s]+)",
        # Acronyms in parentheses
        r"([A-Z][A-Za-z\s&,]+\s*\([A-Z]{2,}\))",
    ]
    
    for pattern in org_patterns:
        matches = re.finditer(pattern, text)
        for match in matches:
            entity = match.group(1).strip()
            if len(entity) > 5 and len(entity) < 100:  # Basic validation
                entities.append(entity)
    
    # Strategy 3: Look for vertical lists near "prepared by" type phrases
    list_context_pattern = r"(?:prepared\s+by|authors?|contributors?)[:\s]*\n+((?:[^\n]+\n+){1,10})"
    matches = re.finditer(list_context_pattern, text, re.IGNORECASE)
    for match in matches:
        list_content = match.group(1)
        items = extract_list_items(list_content)
        entities.extend(items)
    
    # Clean and deduplicate
    cleaned_entities = []
    seen = set()
    for entity in entities:
        entity = clean_entity_name(entity)
        if entity and entity.lower() not in seen and len(entity) > 3:
            seen.add(entity.lower())
            cleaned_entities.append(entity)
    
    return cleaned_entities

def extract_list_items(text):
    """Extract individual items from a list-like text."""
    items = []
    
    # Split by common list delimiters
    lines = text.split('\n')
    for line in lines:
        line = line.strip()
        # Remove bullet points, numbers, etc.
        line = re.sub(r'^[\s•■▪→\-\*\d\.\)]+', '', line).strip()
        if line and len(line) > 3:
            # Check if line contains organization-like text
            if any(suffix in line for suffix in ['Inc', 'LLC', 'Corp', 'Associates', 'Consulting', 'Engineering', 'Agency', 'Department']):
                items.append(line)
            elif re.match(r'^[A-Z]', line):  # Starts with capital letter
                items.append(line)
    
    return items

def clean_entity_name(entity):
    """Clean and standardize entity names."""
    # Remove extra whitespace
    entity = ' '.join(entity.split())
    # Remove trailing punctuation
    entity = entity.rstrip('.,;:')
    # Remove "and" at the end
    entity = re.sub(r'\s+and\s*$', '', entity, flags=re.IGNORECASE)
    return entity

def extract_entities_combined(text, qa_pipeline, questions, confidence_threshold=0.1):
    """
    Combine QA and pattern-based extraction for better coverage.
    """
    all_entities = []
    
    # Get QA-based entities
    qa_entities = extract_entities_from_text(text, questions, qa_pipeline, confidence_threshold, top_k=10)
    for entity in qa_entities:
        all_entities.append({
            'method': 'qa',
            'entity': entity['answer'],
            'confidence': entity['confidence'],
            'question': entity['question']
        })
    
    # Get pattern-based entities
    pattern_entities = extract_entities_enhanced(text)
    for entity in pattern_entities:
        all_entities.append({
            'method': 'pattern',
            'entity': entity,
            'confidence': 0.8,  # Assign a default confidence for pattern matches
            'question': 'Pattern-based extraction'
        })
    
    # Deduplicate while keeping track of methods
    final_entities = {}
    for item in all_entities:
        key = item['entity'].lower().strip()
        if key not in final_entities:
            final_entities[key] = item
        else:
            # Combine methods if found by both
            if final_entities[key]['method'] != item['method']:
                final_entities[key]['method'] = 'both'
                final_entities[key]['confidence'] = max(final_entities[key]['confidence'], item['confidence'])
    
    return list(final_entities.values())

# Test the enhanced extraction
if documents:
    sample_text = documents[0]['pages'][0][:2000] if documents[0]['pages'] else ""
    print("Testing enhanced pattern extraction:")
    pattern_entities = extract_entities_enhanced(sample_text)
    print(f"Found {len(pattern_entities)} entities via patterns:")
    for entity in pattern_entities[:10]:
        print(f"  - {entity}")

## Alternative Approach: Pattern-Based Extraction

As a complement to the QA approach, we can also use pattern matching to find common authorship patterns.

In [ ]:
# Common patterns for identifying document preparers
AUTHORSHIP_PATTERNS = [
    r"prepared by:?\s*([^\n]+)",
    r"prepared for .+ by:?\s*([^\n]+)",
    r"authors?:?\s*([^\n]+)",
    r"submitted by:?\s*([^\n]+)",
    r"consultant:?\s*([^\n]+)",
    r"engineering:?\s*([^\n]+)",
    r"([A-Z][\w\s&,]+(?:Inc\.|LLC|Corp\.|Corporation|Associates|Consulting|Engineers))"
]

def extract_entities_by_pattern(text):
    """Extract entities using regex patterns."""
    if pd.isna(text):
        return []
    
    entities = []
    text_lower = text.lower()
    
    for pattern in AUTHORSHIP_PATTERNS:
        matches = re.finditer(pattern, text, re.IGNORECASE | re.MULTILINE)
        for match in matches:
            entity = match.group(1).strip()
            if len(entity) > 3 and len(entity) < 100:  # Basic validation
                entities.append(entity)
    
    return list(set(entities))  # Remove duplicates

# Test pattern extraction on a sample
if len(documents) > 0:
    sample_text = documents[documents['admin'] == True]['text'].iloc[0]
    pattern_entities = extract_entities_by_pattern(sample_text[:2000])
    print("Entities found by pattern matching:")
    for entity in pattern_entities[:5]:
        print(f"  - {entity}")